<a id='gen-expr'></a>

## 8. 🧩 Pattern 8: Generator Expressions — (expr for x in iterable) — LC 215, 295, 347, 560, 974

---

```
PROBLEM:
  LC 215 — Kth Largest: sum/aggregation over large element streams
  LC 295 — Find Median: running aggregation, no full materialization needed
  LC 347 — Top K Frequent: sum of frequency values
  LC 560 — Subarray Sum Equals K: sum over prefix differences
  LC 974 — Subarray Sums Divisible by K: sum over modular remainders

SYNTAX:
  (expr for x in iterable)           ← lazy generator — yields one item at a time
  (expr for x in iterable if cond)   ← filtered generator
  [expr for x in iterable]           ← list comp — eagerly builds full list in RAM

MEMORY — generator vs list comp:
  import sys
  nums = range(10**6)
  list_comp = [x*x for x in nums]        # allocates ~8 MB list in RAM
  gen_expr  = (x*x for x in nums)        # allocates ~112 bytes — just the iterator
  sys.getsizeof(list_comp)   → ~8 000 056 bytes
  sys.getsizeof(gen_expr)    → ~112 bytes

LAZY EVALUATION — items produced on demand:
  gen = (x*x for x in range(5))
  next(gen)   → 0     ← only 0*0 computed
  next(gen)   → 1     ← only 1*1 computed
  next(gen)   → 4     ← only 2*2 computed
  (the rest are never computed until asked)

PASS DIRECTLY INTO AGGREGATORS — no extra []:
  ✅  sum(x*x for x in nums)         ← generator feeds sum() lazily
  ❌  sum([x*x for x in nums])       ← list built first, then summed — wasted RAM
  ✅  max(len(w) for w in words)     ← no intermediate list
  ✅  any(x > 10 for x in nums)      ← short-circuits on first True — super fast
  ✅  all(x > 0 for x in nums)       ← short-circuits on first False

WHEN TO PREFER GENERATOR OVER LIST COMP:
  Use generator when:
    - feeding result directly into sum/min/max/any/all/next
    - data is large — no need to hold all values at once
    - you only iterate once
  Use list comp when:
    - you need random access (result[i])
    - you iterate the result more than once
    - you need len() of the result

SLOW MOTION TRACE — sum(x*x for x in range(4)):
  aggregator pulls from generator one at a time:
  pull 1: x=0 → yields 0  → running sum = 0
  pull 2: x=1 → yields 1  → running sum = 1
  pull 3: x=2 → yields 4  → running sum = 5
  pull 4: x=3 → yields 9  → running sum = 14
  generator exhausted → sum returns 14
  list never allocated.

KEY INSIGHT:
  Generator expressions are list comps with the brackets swapped for parens.
  Same syntax, zero RAM cost. When the consumer is an aggregator, always drop the [].

TIME / SPACE:
  Time:  O(n) — same as list comp, one pass
  Space: O(1) — constant, regardless of n (vs O(n) for list comp)
```

In [ ]:
# Pattern 8: Generator Expressions
# Same syntax as list comp, parens instead of brackets — zero RAM cost.

import sys

# 1. memory contrast — generator vs list comp
n = 10**6
list_comp = [x * x for x in range(n)]       # eagerly builds full list
gen_expr  = (x * x for x in range(n))       # lazy — just an iterator object
print(f"list comp size : {sys.getsizeof(list_comp):>12,} bytes")
print(f"gen expr size  : {sys.getsizeof(gen_expr):>12,} bytes")

# 2. lazy evaluation — items pulled one at a time
gen = (x * x for x in range(5))
print(f"next() pull 1  : {next(gen)}")   # 0*0=0  — only this computed so far
print(f"next() pull 2  : {next(gen)}")   # 1*1=1
print(f"next() pull 3  : {next(gen)}")   # 2*2=4

# 3. pass directly into aggregators — no intermediate list
nums = range(1, 6)                           # [1, 2, 3, 4, 5]
print(f"sum of squares : {sum(x*x for x in nums)}")          # 55
print(f"max word len   : {max(len(w) for w in ['apple', 'fig', 'banana'])}")  # 6
print(f"any > 100      : {any(x > 100 for x in nums)}")      # False
print(f"all > 0        : {all(x > 0 for x in nums)}")        # True

# 4. rewrite drill — list comp + sum() → single generator expression
data = [3, -1, 4, -1, 5, 9, -2, 6]

# BEFORE (bad — allocates temp list):
positives_list = [x for x in data if x > 0]
total_before   = sum(positives_list)

# AFTER (good — no temp list):
total_after = sum(x for x in data if x > 0)

print(f"sum positives (list comp) : {total_before}")
print(f"sum positives (gen expr)  : {total_after}")


def subarray_sum_equals_k(nums: list, k: int) -> int:
    """
    LC 560 — Subarray Sum Equals K
    Approach: prefix sum hash map; count subarrays where prefix[j]-prefix[i]==k.
    Generator expression used for the running prefix aggregation pattern.
    Args:
        nums (list[int]): integer array.
        k (int): target subarray sum.
    Returns:
        int: count of subarrays with sum equal to k.
    Time:  O(n) — single pass with prefix map
    Space: O(n) — prefix count map
    """
    from collections import defaultdict
    prefix_count = defaultdict(int)
    prefix_count[0] = 1          # empty prefix — sum 0 seen once before start
    prefix_sum = 0
    count = 0

    # slow motion on nums=[1,1,1], k=2:
    # i=0: prefix=1  need=1-2=-1  count+=prefix_count[-1]=0  map={0:1,1:1}
    # i=1: prefix=2  need=2-2=0   count+=prefix_count[0]=1   map={0:1,1:1,2:1}
    # i=2: prefix=3  need=3-2=1   count+=prefix_count[1]=1   map={0:1,1:1,2:1,3:1}
    # result: 2

    for num in nums:
        prefix_sum += num              # grow the running prefix sum
        need = prefix_sum - k          # what prefix sum would complete a subarray of sum k?
        count += prefix_count[need]    # how many times have we seen that prefix?
        prefix_count[prefix_sum] += 1  # record this prefix for future lookups

    return count


def test_harness(fn):
    tests = [
        ([1, 1, 1],    2,  2),    # [1,1] twice
        ([1, 2, 3],    3,  2),    # [1,2] and [3]
        ([1, -1, 0],   0,  3),    # [1,-1], [0], [1,-1,0]
        ([1],          1,  1),    # single match
        ([1],          2,  0),    # no match
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(subarray_sum_equals_k)

# generator aggregation patterns — one-liners
words = ["apple", "fig", "banana", "cherry"]
print(f"longest word   : {max(words, key=lambda w: len(w))}")   # banana
print(f"total chars    : {sum(len(w) for w in words)}")         # 22
print(f"any 3-letter   : {any(len(w) == 3 for w in words)}")   # True (fig)
print(f"all non-empty  : {all(len(w) > 0 for w in words)}")    # True

print("generator_expressions defined.")